In [ ]:
from tqdm import tqdm
from models.sentiment import tokenizer, sentiment
from models.topic import TopicAnalyzer
from data import df
import pandas as pd
import numpy as np

tqdm.pandas() # funkcja umożliwiająca użycie paska postępu w metodzie apply

In [ ]:
df["token_count"] = df["lyrics"].apply(
    lambda x: len(tokenizer.tokenize(x))
)

df.head()

In [ ]:
sentiment_results = (
    df["lyrics"]
    .progress_apply(sentiment)
    .apply(pd.Series)
    .rename(
        columns={
            "negative": "sentiment_negative",
            "neutral": "sentiment_neutral",
            "positive": "sentiment_positive"
        }
    )
)

sentiment_columns = [
    "sentiment_negative",
    "sentiment_neutral",
    "sentiment_positive",
    "sentiment"
]

df[sentiment_columns] = sentiment_results[sentiment_columns]
df.head()

In [ ]:
topic_analyzer = TopicAnalyzer(min_cluster_size=16)

topics, probs = topic_analyzer.fit(df["lyrics"].tolist())

topic_analyzer.save("models/bertopic_model")

df["topic"] = topics

In [ ]:
info = topic_analyzer.get_topic_info()

print(
    f"topics={(info['Topic'] != -1).sum()}, "
    f"outliers={(np.array(topics) == -1).sum()}"
)

In [ ]:
df.to_excel("data/results.xlsx", index=False)
df.to_csv("data/results.csv", index=False)